In [1]:
import pandas as pd
import numpy as np

# Load the training and testing datasets
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display the first few rows of the training and testing datasets
print("Training Data:")
print(train_df.head())
print("\nTesting Data:")
print(test_df.head())

# Display the summary statistics of the training and testing datasets
print("\nTraining Data Summary:")
print(train_df.describe())
print("\nTesting Data Summary:")
print(test_df.describe())

# Distinguish column types
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns

print("\nNumeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Correlation matrix for numeric columns
correlation_matrix = train_df[numeric_cols].corr()
print("\nCorrelation Matrix:")
print(correlation_matrix)

# Visualize the correlation matrix
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numeric Features')
plt.show()

# Visualize the distribution of categorical columns
for col in categorical_cols:
    plt.figure(figsize=(8, 6))
    sns.countplot(data=train_df, x=col)
    plt.title(f'Distribution of {col}')
    plt.xticks(rotation=45)
    plt.show()


Training Data:
    id  bone_length  rotting_flesh  hair_length  color    type
0  472     0.681615       0.529227     0.625242  white   Ghoul
1  170     0.480836       0.407930     0.539005  clear  Goblin
2  189     0.375197       0.742953     0.320764   blue   Ghost
3  861     0.626017       0.172182     0.408422   blue   Ghoul
4   30     0.250770       0.246258     0.554654  black   Ghost

Testing Data:
    id  bone_length  rotting_flesh  hair_length  color    type
0  779     0.516004       0.527508     0.354857  white   Ghoul
1   72     0.523729       0.318483     0.330146  green  Goblin
2   29     0.500197       0.438418     0.532530  clear   Ghoul
3  745     0.417300       0.377595     0.541834  clear  Goblin
4  119     0.515275       0.582627     0.568721  clear  Goblin

Training Data Summary:
               id  bone_length  rotting_flesh  hair_length
count  296.000000   296.000000     296.000000   296.000000
mean   453.574324     0.433147       0.513055     0.525371
std    260.51

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-09-15 07:52:59.041 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['color', 'type'], 'Numeric': ['id', 'bone_length', 'rotting_flesh', 'hair_length'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, MinMaxScale

# Copy the DataFrames to avoid modifying the original data
train_df_copy = train_df.copy()
test_df_copy = test_df.copy()

# Handle missing values
numeric_cols = train_df_copy.select_dtypes(include=[np.number]).columns
categorical_cols = train_df_copy.select_dtypes(include=['object', 'category']).columns

# Fill missing values for numeric columns with mean
fill_missing_numeric = FillMissingValue(features=numeric_cols, strategy='mean')
train_df_copy = fill_missing_numeric.fit_transform(train_df_copy)
test_df_copy = fill_missing_numeric.transform(test_df_copy)

# Fill missing values for categorical columns with most frequent value
fill_missing_categorical = FillMissingValue(features=categorical_cols, strategy='most_frequent')
train_df_copy = fill_missing_categorical.fit_transform(train_df_copy)
test_df_copy = fill_missing_categorical.transform(test_df_copy)

# Encode categorical variables using label encoding
label_encode = LabelEncode(features=categorical_cols)
train_df_copy = label_encode.fit_transform(train_df_copy)
test_df_copy = label_encode.transform(test_df_copy)

# Normalize numerical features using Min-Max scaling
min_max_scale = MinMaxScale(features=numeric_cols)
train_df_copy = min_max_scale.fit_transform(train_df_copy)
test_df_copy = min_max_scale.transform(test_df_copy)

# Display the preprocessed data
print("Preprocessed Training Data:")
print(train_df_copy.head())
print("\nPreprocessed Testing Data:")
print(test_df_copy.head())


Preprocessed Training Data:
         id  bone_length  rotting_flesh  hair_length  color  type
0  0.525670     0.820910       0.469621     0.566954      6     1
1  0.188616     0.555319       0.310079     0.467303      3     2
2  0.209821     0.415579       0.750734     0.215119      2     0
3  0.959821     0.747365       0.000000     0.316410      2     1
4  0.032366     0.250987       0.097431     0.485387      0     0

Preprocessed Testing Data:
         id  bone_length  rotting_flesh  hair_length  color  type
0  0.868304     0.601839       0.467359     0.254515      6     1
1  0.079241     0.612057       0.192429     0.225960      4     2
2  0.031250     0.580929       0.350180     0.459822      3     1
3  0.830357     0.471273       0.270179     0.470573      3     2
4  0.131696     0.600875       0.539857     0.501642      3     2


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df_copy)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'bone_length', 'rotting_flesh', 'hair_length', 'color', 'type'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from metagpt.tools.libs.data_preprocess import LabelEncode

# Assuming train_df_copy and test_df_copy are already preprocessed
X_train = train_df_copy.drop(columns=['type'])
y_train = train_df_copy['type']
X_test = test_df_copy.drop(columns=['type'])
y_test = test_df_copy['type']

# Initialize and train the RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy:.2f}")

# Save the predictions to a CSV file
predictions_df = pd.DataFrame({'id': test_df['id'], 'predicted_type': y_pred})
predictions_df.to_csv('monster_type_predictions.csv', index=False)
print("Predictions saved to 'monster_type_predictions.csv'")


Model Accuracy on Test Set: 0.60
Predictions saved to 'monster_type_predictions.csv'


In [6]:
from sklearn.metrics import accuracy_score

# Assuming the model and preprocessed data are already available from previous tasks
# X_test and y_test are already defined in the previous task

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy:.2f}")

# Save the predictions to a CSV file
predictions_df = pd.DataFrame({'id': test_df['id'], 'predicted_type': y_pred})
predictions_df.to_csv('monster_type_predictions.csv', index=False)
print("Predictions saved to 'monster_type_predictions.csv'")


Model Accuracy on Test Set: 0.60
Predictions saved to 'monster_type_predictions.csv'


In [7]:
# Load the pre-trained model and test data
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Assuming the model is already trained and saved
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Load the test data
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/test.csv'
test_df = pd.read_csv(test_path)

# Preprocess the test data
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, MinMaxScale

numeric_cols = test_df.select_dtypes(include=[np.number]).columns
categorical_cols = test_df.select_dtypes(include=['object', 'category']).columns

fill_missing_numeric = FillMissingValue(features=numeric_cols, strategy='mean')
test_df = fill_missing_numeric.transform(test_df)

fill_missing_categorical = FillMissingValue(features=categorical_cols, strategy='most_frequent')
test_df = fill_missing_categorical.transform(test_df)

label_encode = LabelEncode(features=categorical_cols)
test_df = label_encode.transform(test_df)

min_max_scale = MinMaxScale(features=numeric_cols)
test_df = min_max_scale.transform(test_df)

# Prepare the test features
X_test = test_df.drop(columns=['type'])
y_test = test_df['type']

# Predict using the trained model
y_pred = model.predict(X_test)

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy:.2f}")

# Save predictions to a CSV file
predictions_df = pd.DataFrame({'id': test_df['id'], 'predicted_type': y_pred})
predictions_df.to_csv('monster_type_predictions.csv', index=False)
print("Predictions saved to 'monster_type_predictions.csv'")


NotFittedError: This SimpleImputer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.